# <center>Hello<center>

#### <center>I'm not really sure what I'm doing, but I guess we will find out <center>

# Sensitivity Calculations

In [82]:
SPEED_OF_LIGHT = 2.99792458e8 #m/s
TARGET_FREQUENCY =1.42e9 #Hz
TARGET_WAVELENGHT = SPEED_OF_LIGHT/TARGET_FREQUENCY
BANDWIDTH = 1.95e3 #Hz
# BANDWIDTH_KMS = 200#kms

APERTURE_DIAMETER = 0.185 #m

Antenna 

In [53]:
BEAM_WIDTH = 0.89*TARGET_WAVELENGHT/APERTURE_DIAMETER #radian
EFFECTIVE_AREA_OF_ANTENNA = TARGET_WAVELENGHT**2/(4*np.pi)
BEAM_SOLID_ANGLE = BEAM_WIDTH**2
# BEAM_SOLID_ANGLE_IF_IN_DEGREES = (BEAM_WIDTH*np.pi/180)**2
PEAK_GAIN = (4*np.pi)/BEAM_SOLID_ANGLE
EFFECTIVE_COLLECTING_AREA = TARGET_WAVELENGHT**2/BEAM_SOLID_ANGLE


Source


In [54]:
LAT_1=105
LAT_2=88
SOURCE_RADIUS_IN_DEGREES = LAT_1-LAT_2
TARGET_1_SOLID_ANGLE = (SOURCE_RADIUS_IN_DEGREES*np.pi/180)**2
BRIGHTNESS_TEMP_HI_TARGET_1 = 120 #lobe 
BRIGHTNESS_TEMP_HI_TARGET_2 = 100 #centre



In [56]:
ANTENNA_TEMP = EFFECTIVE_COLLECTING_AREA*BRIGHTNESS_TEMP_HI_TARGET_1*TARGET_1_SOLID_ANGLE/TARGET_WAVELENGHT**2

System Noise

In [ ]:
#K
T_CMB = 2.72548 
# T_SUN = not important if not in the observation
T_ATM = 0 #2 #Couldn't find a good enough solid source tried reich & reich 1988
T_SPILL = 0 #? I think this would require a area scan with the telescope
T_RSB = (TARGET_FREQUENCY/1.4e9)**-2.7#Reflective solar bands this is the average background sky brightness so includes atm
HACKRF_ONE_NOISE = 11 #dB
# NOISE_FIGURE_HACKRF_ONE = 10*np.log10(FN)
FN = 10**(HACKRF_ONE_NOISE/10)
T_0 = 290

T_RADIOMETER = (FN-1)*T_0 # gotten during calibration 
T_LNA = 59
DT_SRC = 0.04

GAIN =42 #dB
LINEAR_GAIN = 10**(GAIN/10)

T_sys = T_CMB+T_RSB+DT_SRC+T_ATM+T_SPILL+(T_RADIOMETER/LINEAR_GAIN)+T_LNA+ANTENNA_TEMP
print(T_sys)


#this is acceptable

73.18073156527954


In [70]:
SNR_target = 10

In [ ]:
print(f'T_sys: {T_sys}')
print(f'Antenna_temp: {ANTENNA_TEMP}')
print(f'BANDWIDTH: {BANDWIDTH}')
integration_time = (1/BANDWIDTH)*((SNR_target*T_sys)/(ANTENNA_TEMP))**2
print(f'\N{greek small letter tau}: {integration_time:.4f}seconds')

T_RMS = T_sys/np.sqrt(BANDWIDTH*integration_time)



T_sys: 73.18073156527954
Antenna_temp: 10.240768540169674
BANDWIDTH: 1950.0
τ: 2.6187seconds
1.0240768540169674
10.0


The final numbers are

In [ ]:
print(f'{}')
print(f'{}')
print(f'{}')
print(f'{}')
print(f'{}')

The sensitivity of our experiment is 

In [ ]:
print(f'{}')

# Data Analysis

## Init

This is an init I think for how the plots get displayed

In [1]:
%matplotlib widget

In [17]:
from pathlib import Path
import os
import sys

ROOT_DIR = os.path.dirname(os.path.abspath(''))
DATA_DIR = ROOT_DIR + '/DATA/'
print(DATA_DIR)

/home/matthew/radio_astronomy/assignments/cantenna/Radio_Astronomy_group_5_cantenna/DATA/


## The main course

In [24]:
import os
import glob
import matplotlib.pyplot as plt
import numpy as np
import astropy.units as u
from astropy.table import QTable
import astropy.constants as constants
import healpy as hp
import scipy

def read_file(file_name):

    #TODO We rewriting this for our own purpose 
    #TODO I don't know the format of the data: binary?
    #TODO I don't know what format it has to be in 
    #TODO We don't have a velocity, do we need one, if so how calculate

    f = np.fromfile(open(file_name))
    print(f)
    return f

    # r'''
    # Read a HEALPix map frame from the LAB survey.

    # Parameters
    # ----------

    # file_name : string: filename to read.

    
    # Returns
    # -------

    # tuple (astropy quantity, astropy quantity array) representing (mean_velocity, HEALPix_image) in 
    # km/s and K respectively.
    # '''
    # velocity=velocity_from_filename(file_name)*u.km/u.s
    # return (velocity, hp.read_map(file_name))



In [25]:

data2 = read_file(Path(DATA_DIR) / 'data2')

[2.22593860e-12 1.50807619e-18 1.40890229e-15 ... 3.27363035e-22
 3.07420753e-33 7.92681076e-26]


In [ ]:
print(data2.shape)
data_reshaped = data2.reshape(-1,1024)
print(data_reshaped.shape)


(54067200,)
(52800, 1024)


In [ ]:

#def gaussian_beam_fn(fwhm=70*u.deg):
#    def fn(angular_distance):
#        d = angular_distance    
#        return np.exp(-(d.to(u.rad).value**2/(2*(fwhm.to(u.rad).value/2.355)**2)))
#    return fn


def airy_beam_fn(r_to_first_null=70*u.deg):
    r'''
    Returns a function(angular_distance) that returns the power beam gain of the airy pattern
    with peak-to-first-null given as a parameter to THIS function size at a certain angular distance.
    
    '''
    r_null = r_to_first_null.to(u.rad)/2
    def fn(angular_distance):
        d = angular_distance
        l = d.to(u.rad)*1.9158715
        return (2*scipy.special.j1(l/r_null)/(l/r_null))**2
    return fn




#TODO not sure if we can use the map from this class but the spectra yeah sure

class LABSurvey:
    def __init__(self, root_dir):
        r'''
        Plot average spectra over parts of the sky. Survey can be downloaded from:
            https://lambda.gsfc.nasa.gov/data/foregrounds/HI/lab_healpix.tar.bz2
        
        Parameters
        ----------
        root_dir: string: path containing the HEALPIX FITS files of the LAB survey.

        Variables
        ---------
        
        images : 2D array of astropy quantities: the actual HEALPix maps. Indices: [frame, pixel]
        velocity : Array of astropy quantities, length number of frames, contains mean velocity
                   of each frame in km/s
        frequency : Array of astropy quantities, length number of frames, contains mean frequency 
                    of each frame in MHz.
        
        '''
        file_names = glob.glob(os.path.join(root_dir)+'*cut*.fits')
        self.nside = 512
        images = [read_file(name) for name in file_names]
        images.sort(key=lambda x: x[0])
        self.velocity = np.array([x[0].value for x in images])*u.km/u.s
        f0 = 1420.405751768*u.MHz
        self.frequency = (f0 - f0*self.velocity/constants.c).to(u.MHz)
        self.images = np.array([x[1] for x in images])*u.K


    def plot_map(self, map_index, **args):
        return hp.mollview(map=self.images[map_index], unit='K', title="Velocity: %s" % (self.velocity[map_index],), **args)


    def plot_beam_map(self, map_index, lon, lat, beam_size=70*u.deg, **args):
        beam_fn = airy_beam_fn(beam_size)
        vec = hp.ang2vec(theta=lon.to(u.deg).value, phi=lat.to(u.deg).value,
                         lonlat=True)
        mask = hp.query_disc(nside=self.nside, vec=vec, 
                             radius=np.pi)
        angular_distances = hp.rotator.angdist(vec, hp.pix2vec(nside=self.nside, ipix=mask))*u.rad
        beam = np.squeeze(beam_fn(angular_distances))
        image = self.images[map_index].copy()
        image[mask] *= beam
        return hp.mollview(map=image, unit='K', title="Velocity: %s" % (self.velocity[map_index],), **args)

    
    def spectrum(self, lon, lat, beam_fn=airy_beam_fn(70*u.deg), radius=180*u.deg):
        r'''
        '''
        vec = hp.ang2vec(theta=lon.to(u.deg).value, phi=lat.to(u.deg).value,
                         lonlat=True)
        mask = hp.query_disc(nside=self.nside, vec=vec, 
                             radius=min(radius.to(u.rad).value, np.pi))
        angular_distances = hp.rotator.angdist(vec, hp.pix2vec(nside=self.nside, ipix=mask))*u.rad
        beam = np.squeeze(beam_fn(angular_distances))
        return (self.images[:,mask]*beam[np.newaxis,:]).mean(axis=1)/beam.mean()

    
    def plot_spectrum(self, ax, lon, lat, beam_size=70*u.deg, beam_fn=airy_beam_fn, x_axis='velocity'):
        r'''
        Parameters
        ----------
        ax : Matplotlib Axis instance to use for plotting.
        lon : Astropy angular quantity: galactic longitude l of pointing centre, e.g. 85*u.deg.
        lat : Astropy angular quantity: galactic latitude b of pointing centre, e.g. 85*u.deg.
        beam_size : Astropy angular quantity: approximate FWHM of beam, assuming a Gaussian beam shape
                    or peak-to-first-null when using the airy pattern beam.
        beam_fn : The beam shape function. Either gaussian_beam_fn or airy_beam_fn.
        x_axis : String: 'velocity' or 'frequency'
        '''
        avg_spectrum = self.spectrum(lon=lon, lat=lat,
                                beam_fn=beam_fn(beam_size),
                                radius=3*beam_size)
        if x_axis=='velocity':
            ax.plot(self.velocity, avg_spectrum)
            ax.set_xlabel('Radio radial velocity [km/s]')
        elif x_axis=='frequency':
            ax.plot(self.frequency, avg_spectrum)
            ax.set_xlabel('Frequency [MHz]')
        else:
            raise ValueError("x_axis must be 'velocity' or 'frequency', not '%s'" % x_axis)
        ax.grid()
        
        ax.set_ylabel('Brightness temperature [K]')
        ax.set_title(r'Neutral Hydrogen $T_\mathrm{B}$ at $l,b$ = %.2f$^\circ$,%.2f$^\circ$; size %.2f$^\circ$ ' % 
                     (lon.to(u.deg).value, lat.to(u.deg).value, beam_size.to(u.deg).value))

# Instantiate the object that contains the entire survey.
LAB = LABSurvey(ROOT_DIR)


In [ ]:
frame_units = QTable(data=[np.arange(LAB.velocity.shape[0]),LAB.velocity, LAB.frequency],
                    names=['Index', 'Velocity', 'Frequency'])

print('\n'.join(frame_units.pformat(max_lines=-1)))

In [ ]:
LAB.plot_map(45, min=0, max=100)

In [ ]:
LAB.plot_beam_map(45, lon=75*u.deg, lat=0*u.deg, beam_size=20*u.deg, min=0, max=100)

In [ ]:
fig, (ax_v, ax_f) = plt.subplots(2,1,figsize=(6,6),dpi=200)

lon=175*u.deg
lat=31.5*u.deg
beam_size=1*u.deg

LAB.plot_spectrum(ax_v,lon=lon, lat=lat, beam_size=beam_size, beam_fn=airy_beam_fn,
                  x_axis='velocity')

LAB.plot_spectrum(ax_f,lon=lon, lat=lat, beam_size=beam_size, beam_fn=airy_beam_fn,
                  x_axis='frequency')

fig.subplots_adjust(hspace=0.4)